# Introduction 

The Glasgow Royal Infirmary (GRI) cohort is a retrospective colorectal cancer (CRC) cohort of stage I-III patients from Prof. Joanne Edward's lab. We are interesting in modelling the clinical data associated with these patients in order to predict cancer recurrence and survival probabilities of patients with CRC.

Below there is the dataset and the metadata we have collected with the help of Joanne Edward's lab.

# Dataset

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns 
import matplotlib.pyplot as plt
import missingno as msno
from sksurv.util import Surv
from sksurv.linear_model import CoxPHSurvivalAnalysis
pd.set_option("display.max_columns", None)


In [ ]:
df = pd.read_excel("data/GRI_FullDataset.xlsx", na_values=["NaN", "nan", "NA", "NAN"])

In [ ]:
df

In [ ]:
print(df.isna().sum().to_string())


In [ ]:
# Plot missigness
# Bar chart (missing counts per column)
msno.bar(df)
plt.show()

# Heatmap of missing-value correlations
#msno.heatmap(df)
#plt.show()

plt.figure(figsize=(20, 8))   # make the figure wide
sns.heatmap(df.isna(), cbar=False, cmap="viridis")

plt.xticks(rotation=90, fontsize=8)   # rotate column labels
plt.yticks([])                        # hide row labels (too many)
plt.show()

Initial preprocessing includes exclusion criteria such not including those patients that have neoadjuvant therapy. 
Also, some patients gave the value 999 as NA, and the event is set as "12", instead of 0, 1 or 2.

In [ ]:
df["UpdateJuly2020_a0_cd1_ncd2"].value_counts(dropna=False)

In [ ]:
# looking at the one with 12 as value 
df[df["UpdateJuly2020_a0_cd1_ncd2"] == 12]

# it is commented in Notes that the person moved. It is probably alive when that happened and so the most appropiate is a 1 (alive) as status. 
# Also REC_no0_yes1 suggests that there is no recurrence
df.loc[df["UpdateJuly2020_a0_cd1_ncd2"] == 12, "UpdateJuly2020_a0_cd1_ncd2"] = 1


In [ ]:
# Detecting some value of "999" what is inserted when NA
(df["neoadj1"] == 999).sum()

df["neoadj1"].value_counts(dropna=False)

print((df == 999).sum().to_string())

In [ ]:
# replace it with NA
df.replace(999, np.nan, inplace=True)


In [ ]:
# make a subset with those that contain neoadjuvant therapy 0
(df["neoadj1"] == 1).sum()
df = df[df["neoadj1"] == 0]

In [ ]:
df

In [ ]:
## make the dates
data = df.copy()

# Convert dates from "dd.mm.yyyy" format
data["Date_of_scan"] = pd.to_datetime(df["Date_of_scan"], format="%d.%m.%Y", errors="coerce")
data["DateofLastFollowUp01072020"] = pd.to_datetime(df["DateofLastFollowUp01072020"], format="%d.%m.%Y", errors="coerce")
data["date_recurrence_01072020"] = pd.to_datetime(df["date_recurrence_01072020"], format="%d.%m.%Y", errors="coerce")
data["dofsurg"] = pd.to_datetime(df["dofsurg"], format="%d.%m.%Y", errors="coerce")
data["dob"] = pd.to_datetime(df["dob"], format="%d.%m.%Y", errors="coerce")

In [ ]:
# make the operation labels correctly
data["operation"] = data["operation"].astype(str).str.upper()

data["operation"] = data["operation"].str.replace(r"EXT RIGH", "EXT RT", regex=True)
data["operation"] = data["operation"].str.replace(r"EXT RH", "EXT RT", regex=True)

data["operation"] = data["operation"].str.replace(r"PROCTOCO", "PANPROCT", regex=True)
data["operation"] = data["operation"].str.replace(r"PROCTECT", "PANPROCT", regex=True)

data["operation"] = data["operation"].str.replace(r"(?<!\w)APR\s*\?(?!\w)", "APR", regex=True)

print(data["operation"].unique())


In [ ]:

df2 = data.copy()

# Condition for NA rows in any of these columns
na_mask = df2[["Recurrence_local1_distant2", "REC_n0_yes1", "Colin_full_recurrence_sites"]].isna().any(axis=1)

# Apply the same logic as in R
df2.loc[na_mask, "Recurrence_local1_distant2"] = np.where(
    df2.loc[na_mask, "REC_n0_yes1"] == 0,
    0,
    df2.loc[na_mask, "Recurrence_local1_distant2"]
)

df2.loc[na_mask, "Colin_full_recurrence_sites"] = np.where(
    df2.loc[na_mask, "REC_n0_yes1"] == 0,
    0,
    df2.loc[na_mask, "Colin_full_recurrence_sites"]
)

df2.loc[na_mask, "REC_n0_yes1"] = np.where(
    df2.loc[na_mask, "Recurrence_local1_distant2"] == 0,
    0,
    df2.loc[na_mask, "REC_n0_yes1"]
)

# Create a new label for Recurrence_local1_distant2, if the label is == to 4, means that it had recurrence and we do not know where. 
na_mask2 = df2["Recurrence_local1_distant2"].isna()

df2.loc[na_mask2, "Recurrence_local1_distant2"] = np.where(
    df2.loc[na_mask2, "REC_n0_yes1"] == 1,
    4,
    df2.loc[na_mask2, "Recurrence_local1_distant2"]
)



### Basic demographic 

In [ ]:
df2["sex"].value_counts(normalize=True) * 100

In [ ]:
df2["age"].mean()

In [ ]:
## super important to calculate the ratio of limph nodes that are positive
df2["LNratio"] = np.where( df2["LNsample"] == 0, 0, (df2["LNmet"] / df2["LNsample"]).round(3))

### Variables for competing risks

Basically it is the same as the event indicator, but if they had recurrence, they are treated like if they died by other causes.
It requires the last follow up information recorded in "UpdateJuly2020_a0_cd1_ncd2", and the Disease free status information from "DFS_status_July2020".
If it has had recurrence meaning DFS_status_July == 1, then that is our event of interest now.

Recurrence/died of cancer = event of interest

Alive without recurrence = censor

Died of other causes without recurrece = competing event



In [ ]:
df2["Cmprsk_Status"] = df2["UpdateJuly2020_a0_cd1_ncd2"]
df2["Cmprsk_Status"] = np.where(df2["DFS_status_July2020"] == 1, 1,  df2["Cmprsk_Status"])

In [ ]:
df2["Cmprsk_Status"].value_counts(normalize=True) * 100

It is not easy to define the event of interest, since we have cases where they might have died of other causes but they have had recurrence as well. 
Since we are primarily interested in recurrence, we have to prioritize it. Therefore, if it has had recurrence, we would assign it the event of interest regardless of the other cause of death.

Some examples where the overall status, does not correspond with the disease free status:


In [ ]:
mask = df2["UpdateJuly2020_a0_cd1_ncd2"] != df2["DFS_status_July2020"]
df2.loc[mask, ["UpdateJuly2020_a0_cd1_ncd2", "DFS_status_July2020", "REC_n0_yes1", "Cmprsk_Status"]]

Before we get the complete cases we need to subset the dataset, or perform inputation. 

In [ ]:
complete_cases = df2.dropna(axis = 0, how = 'any', inplace = True)
print(complete_cases)



# Quality Performance Indicators (QPIs)
## Colorectal cancer quality performance indicator (QPI) documentation

https://publichealthscotland.scot/population-health/conditions-and-diseases/cancer/cancer-quality-performance-indicators-qpis/quality-performance-indicators-qpis/qpi-documentation/


https://publichealthscotland.scot/publications/colorectal-cancer-quality-performance-indicator-qpi-documentation/colorectal-cancer-quality-performance-indicator-qpi-documentation-1-april-2021-onwards/

01 May 2023 (Latest release)

National cancer quality performance indicators have been developed to support continuous quality improvement in cancer care (CEL 06 2012). NHS Boards are required to report these indicators against a clinically agreed indicator specific target as part of the mandatory national cancer quality programme. They have been developed collaboratively by North Cancer Alliance, South East Scotland Cancer Network, West of Scotland Cancer Network, Healthcare Improvement Scotland and PHS.

They are recorded to evaluate if patients with colorectal cancer are receiving a high standard of care. 

| **Field Name** | **Description**                                             | **Required for QPI(s)**              |
| -------------- | ----------------------------------------------------------- | ------------------------------------ |
| DATEIMAGELB    | Date of Imaging Large Bowel                                 | QPI 2                                |
| DIAGDATE       | Date of Diagnosis (Cancer)                                  | QPI 1, 2, 5, 7, 8, 9, 10, 11, 12, 16 |
| SITEPRIM       | Site of Origin of Primary Tumour (Cancer)                   | QPI 1, 7                             |
| FINSURGDATE    | Date of Final Definitive (or Only) Surgery                  | QPI 2, 7, 10, 11                     |
| PRESENT        | Presentation Type (Elective/Emergency)                      | QPI 1, 2, 7, 10                      |
| OPINTENT       | Intent of Surgery (Curative/Palliative)                     | QPI 1, 2, 5                          |
| REOPER         | Re-operation after definitive surgery                       | QPI 8                                |
| ANASLEAK       | Anastomotic Leak after surgery                              | QPI 9                                |
| CIRCMARGIN     | Circumferential Margin Involved (surgical specimen)         | QPI 7                                |
| LNEXAMINE      | Final Total Number of Lymph Nodes Examined                  | QPI 5                                |
| FINALN         | TNM Nodal Classification (Final)                            | QPI 11                               |
| FINALT         | TNM Tumour Classification (Final)                           | QPI 11                               |
| FINALM         | TNM Metastasis Classification (Final)                       | QPI 11                               |
| NEOONC         | Neo-Adjuvant Oncology Treatment Type                        | QPI 11                               |
| ADJONC         | Primary/Palliative/Adjuvant Oncology Treatment Type         | QPI 1                                |
| ADJONC\_DATE   | Date Primary/Palliative/Adjuvant Oncology Treatment Started | QPI 11                               |
| LIVERDIAGDATE  | Date of Liver Imaging confirming liver mets                 | QPI 15                               |
| HPBMDT         | Referral to HPB MDT (liver mets patients)                   | QPI 15                               |
| BRAF           | BRAF mutation status                                        | QPI 16                               |
| MSISTATUS      | Microsatellite Instability (MSI) Status                     | QPI 16                               |
| IHCSTATUS      | Mismatch Repair (MMR IHC)                                   | QPI 16                               |
| GENETICS       | Genetics Referral                                           | QPI 16                               |
| TRIAL          | Patient entered into Clinical Trial                         | Generic QPIs                         |

The QPI's names are the following:

| **QPI** | **Name**                                                      |
| ------- | ------------------------------------------------------------- |
| QPI 1   | Radiological Diagnosis of Colorectal Cancer                   |
| QPI 2   | Pre-operative Imaging of Colon                                |
| QPI 3   | *Retired* (Staging Laparoscopy – removed in earlier versions) |
| QPI 4   | Pre-operative MRI in Rectal Cancer                            |
| QPI 5   | Lymph Node Yield                                              |
| QPI 6   | Pre-operative Radiotherapy in Rectal Cancer                   |
| QPI 7   | Clear Surgical Margins                                        |
| QPI 8   | Re-operation                                                  |
| QPI 9   | Anastomotic Leak                                              |
| QPI 10  | Emergency Surgery                                             |
| QPI 11  | Pathological Staging                                          |
| QPI 12  | Adjuvant Chemotherapy in Stage III Colon Cancer               |
| QPI 13  | Adjuvant Radiotherapy in Rectal Cancer                        |
| QPI 14  | Adjuvant Chemotherapy in Rectal Cancer                        |
| QPI 15  | Imaging for Liver Metastases                                  |
| QPI 16  | Molecular Pathological Markers                                |

In our dataset we have lots of more variables, some of them can be understood as extenstion of the standard QPI's. For instance the GRI contains more genertic and pathology variables, which in the future could be converted into the standard practise.


In our dataset:

| **GRI column**            | **QPI dataset variable**                    | **QPI No.**                         | **QPI Name**                                                                     |
| ---------------------------------- | ------------------------------------------- | ----------------------------------- | -------------------------------------------------------------------------------- |
| `UpdateJuly2020_a0_cd1_ncd2`       | Treatment update / recurrence status fields | QPI 12                              | Adjuvant Chemotherapy in Stage III Colon Cancer                                  |
| `Thirtyday_mortality_2017`         | 30-day mortality (audit item, not a QPI)    | –                                   | –                                                                                |
| `MMR_Status_Jen`                   | `MSISTATUS` / `IHCSTATUS`                   | QPI 16                              | Molecular Pathological Markers                                                   |
| `BRAF`                   | `BRAF`                   | QPI 16                              | Molecular Pathological Markers                                                   |
| `dofsurg` (date of surgery)        | `FINSURGDATE`                               | QPI 2, 7, 10, 11                    | Definitive Surgery / Surgical Margins / Emergency Surgery / Pathological Staging |
| `operation`                        | `OPINTENT`                                  | QPI 1, 2, 5                         | Surgical Intent (Curative/Palliative)                                            |
| `sex` / `Female` / `Male`          | `SEX`                                       | Background (not QPI)                | –                                                                                |
| `Location` / `Right0_Left1`        | `SITEPRIM`                                  | QPI 1, 7                            | Radiological Diagnosis / Clear Surgical Margins                                  |
| `tstage` / `nstage` / `mstage`     | `FINALT` / `FINALN` / `FINALM`              | QPI 11                              | Pathological Staging                                                             |
| `Differentiation`                  | `DIFFGRD` (if present)                      | Not core QPI (but pathology detail) | –                                                                                |
| `LNmet`, `LNsample`, `less12nodes`, `LNratio` | `LNEXAMINE`                                 | QPI 5                               | Lymph Node Yield                                                                 |
| `MargInv`                          | `CIRCMARGIN`                                | QPI 7                               | Clear Surgical Margins                                                           |
| `Recurrence_local1_distant2`       | Recurrence coding                           | QPI 12–14                           | Adjuvant therapy follow-up                                                       |
| `PrimaryCOD`                       | Cause of death                              | Background (not QPI)                | –                                           |


Additionally we have the BRAF mutation status.

Lots of recurrence and competing risk status do not appear in the QPI's but they are quite useful


Regarding the genetics or QPI 16, we miss MLH1 in GRI. Overall QPI miss lots of important variables like tumor budding etc. 


Let's add the mutation data

In [ ]:
mutations = pd.read_excel("data/Mutations_GRI.xlsx", na_values=["NaN", "nan", "NA", "NAN"])

mutations

In [ ]:
df3 = df2.merge(mutations, on="TMA_order") 

Which of te QPIs can be predictive?

In [ ]:
qpis = [
    #"UpdateJuly2020_a0_cd1_ncd2",
    #"Thirtyday_mortality_2017", # not for prediction and part od DFS_months
    "MMR_Status_Jen",
    "BRAF",
    #"dofsurg", #not for prediction
    #"operation", # operation has causal blind spots
    "sex",
    "Location", "Right0_Left1",
    "tstage", "nstage", "mstage",
    "Differentiation",
    "LNratio",
    "MargInv",
    #"Recurrence_local1_distant2", #not for prediction
    #"PrimaryCOD", # removing this since we gave cmprsk status
    "DFS_months_2020",
    #"DFS_status_July2020", # moved to Cmprsk_Status
    "Cmprsk_Status"
]

df_qpis = df3[qpis]

In [ ]:
df_qpis

In [ ]:
print(df_qpis.isna().sum().to_string())


In [ ]:
#sns.heatmap(df_qpis.isna(), cbar=False, cmap="viridis")

df_qpis_short = df_qpis.dropna(axis = 0)

#print(df_qpis.shape)   


Just as an example... this would require proper missing data inputation.

### Remove columns that we do not need

These are columns highly missing (smoke, weight...), like >50%





In [ ]:
df3["ASA_group"].isna().sum()

In [ ]:
print(df3.columns.to_list())

We can drop some covariates that are coded version of continues variables. There is one coded version per each continous biomarker. 

We can also drop treatment variables, since including them as predictors would cause causal blind spots, immortal time bias, etc. 

Variables used to calculate DFS or competing risks status are also not longer needed. 

Some varibles such weight, height and BMI are highly missing and then we can remove them. 


In [ ]:
df4 = df3.drop(labels=['TMA_number',"CSS_months_2020", "UpdateJuly2020_a0_cd1_ncd2", "neoadj1", "Thirtyday_mortality_2017",
                 "CMS_Raheleh_withoutlabels", "DFS_status_July2020", 'DateofLastFollowUp01072020', 'date_recurrence_01072020', 
                 'dofsurg', 'dob', 'REC_n0_yes1', 'DFS_months', 'DFS_years', 'Notes', 'PrimaryCOD', 'Local_recurrenceKK', 
                 'Distant_recurrenceKK', 'Recurrence_Location', 'Recurrence_code', 'operation', 'Date_of_scan', 
                 'agecd','Female', 'Male','Diffcd', 'Petersen_Index_low0_high1', 'TSP_Pete_HL', 'KM_Pete_HL', 'weight', 'height', 'BMI_code', 'BMIunder30', 
                 'BMIover30Y1N0', 'Smoker_n0_ex1_curr2', 'smoker', 'cigsday', 'exsmoker', 'anysmoke','LNmet', 'LNsample', 'less12nodes',
                 'crp_code', 'alb_code', 'Aspirin', 'NSAID', 'Statin', 'Metformin', 'Mucin', 'NLR_gtr5', 'Neut_gtr75','LNR_code', 'Less_than_10', 
                 'ASCO_high_risk', 'Ueno_CLR_1mmcutoff', 'Vayrynen_CLR_0.38cutoff', 'IE_lymphocytesHL', 'filter_$', 'Ki67_Code_30', 
                 # coded versions of raw and combinations:
                 'S100A2_code', 'combined_tbs100', 'CD3_code', 'CD3_FOXP3_code', 'CD68_code', 'combinedCD3CD68', 'CXCL9_s_code', 'CXCL9_t_code',
                 'CXCL10_s_code', 'CXCL10_t_code', 'cyclind1_code', 'KI67_tumour_code', 'CD163_code', 'CD66b_code',
                 'STAT1_tumour_cutoff_BK',  'CD3FOXP3_n_code', 'Bcat_Cyto_Coded', 'Bcat_nuc_coded', 'Bcat_mem_coded', 'Fascin_cyto_coded', 
                 'Ecad_cyto_coded', 'Ecad_mem_coded', 'Zeb_cyto_coded','Zeb_nuc_coded', 'Twist_cyto_coded', 
                 'Twist_nuc_coded', 'Snail_cyto_coded', 'Snail_nuc_coded', 'SOX9_cyto_coded', 'SOX9_nuc_coded',
                 'SMAD4_Cyto_67.51_coded', 'SMAD4_Nu_6.91_coded', 'SMAD4_CytoNu_combined', 'twolines_SMAD4_Cyto_Nu_combined', 
                 'SMAD4_Stroma_Cyto_4.94_coded', 'SMAD4_Stroma_Nu_3.88_coded', 'SMAD4_Stroma_CytoNu_combined', 'Zeb1_cyto_IE_code', 
                 'Zeb1_cyto_IE_mc_code', 'Bud_histoscore_Zeb1_code', 'Bud_histo_Zeb_mc_code', 'Cyto_budcohort_zeb1_code', 
                 'Nuc_budcohort_zeb1_code', 'pSTAT3_UH_TC_198.01', 'pSTAT3_Tum_Bud_PM'], axis=1)
                

In [ ]:
plt.figure(figsize=(20, 8))   # make the figure wide
sns.heatmap(df4.isna(), cbar=False, cmap="viridis")

plt.xticks(rotation=90, fontsize=8)   # rotate column labels
plt.yticks([])                        # hide row labels (too many)
plt.show()

Also removing those columns where we have more than 50% of missing values, since we will perform MICE. 

In [ ]:

df4.shape


In [ ]:
k = int(np.ceil(0.5 * df4.shape[0]))  # required non-NA per row
k

In [ ]:
df5 = df4.dropna(axis=1, thresh=k)
df5

In [ ]:
plt.figure(figsize=(20, 8))   # make the figure wide
sns.heatmap(df5.isna(), cbar=False, cmap="viridis")

plt.xticks(rotation=90, fontsize=8)   # rotate column labels
plt.yticks([])                        # hide row labels (too many)
plt.show()